# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maison-ux/flyrank-assignament1-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

This is a ranking/scoring task. The goal is to rank content items by their priority for review, so an editor can focus on the pages that show the strongest observable signals of needing attention.


In [10]:
task_type = "ranking/scoring"
decision = "which content items should be reviewed first?"

print("Task type:", task_type)
print("Decision:", decision)


Task type: ranking/scoring
Decision: which content items should be reviewed first?


## 2. Target or proxy

The target/proxy is an observed decline outcome from the content's subsequent trend. The ranking output should prioritize content items using signals available before that outcome is known. The outcome is used for evaluation, not as a prediction feature.

In [11]:
import pandas as pd

data_url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(data_url)

# The target is derived from the observed trend outcome.
df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower().eq("down").astype(int)
)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Declining rate:", round(df["is_declining_label"].mean(), 3))
print("Leakage fields excluded from features:", ["trend_direction", "trend_pct"])

Rows: 30000
Columns: 45
Declining rate: 0.542
Leakage fields excluded from features: ['trend_direction', 'trend_pct']


## 3. Success metric

The success metric is Precision@K, with K=20 and K=50. A good ranking places a high fraction of actually declining content items near the top of the review queue.

In [12]:
def precision_at_k(scores, labels, k):
    order = pd.Series(scores, index=labels.index).sort_values(ascending=False).index[:k]
    return labels.loc[order].mean()

stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)

baseline_score = stale * visible * df["impressions_90d"]

print("Observed decline rate:", round(df["is_declining_label"].mean(), 3))

for k in [20, 50]:
    print(
        f"Baseline Precision@{k}: "
        f"{precision_at_k(baseline_score, df['is_declining_label'], k):.3f}"
    )

Observed decline rate: 0.542
Baseline Precision@20: 0.850
Baseline Precision@50: 0.640


## 4. The unit of analysis, as a real dataframe

One row represents one content item. The dataframe contains one record per content item, with observable content and performance signals that can be used to prioritize which items should be reviewed.

In [13]:
# Show the real unit of analysis.
print("Unit of analysis: one row = one content item")
print("Shape:", df.shape)

display(
    df[
        [
            "content_id",
            "client_id",
            "content_age_days",
            "days_since_last_update",
            "impressions_90d",
            "avg_position",
            "ctr",
            "word_count"
        ]
    ].head(10)
)


Unit of analysis: one row = one content item
Shape: (30000, 45)


,content_id,client_id,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr,word_count
0,content_304f48230142,client_f369cb89fc,187,20,3803,10.6,0.76,3221.0
1,content_a1fb4e703a9e,client_4e07408562,445,25,15320,20.3,0.05,2481.0
2,content_9aa793d4d895,client_7f2253d7e2,141,20,12581,36.5,0.09,3515.0
3,content_331d6c4de07b,client_19581e27de,463,22,11751,6.2,0.49,NaN
4,content_d99b7a2d90ca,client_3fdba35f04,263,14,19140,44.0,0.13,2803.0
5,content_d4084a4bc775,client_f369cb89fc,147,20,3970,8.5,0.03,3080.0
6,content_9a34b442b552,client_8722616204,90,20,20,7.0,0.00,3059.0
7,content_a63219c6e95a,client_19581e27de,445,22,1724,21.2,0.06,NaN
8,content_5e6c160719bc,client_6208ef0f77,90,20,32574,46.0,0.09,3807.0
9,content_c27558df2b0c,client_19581e27de,257,104,1240,4.9,0.16,NaN


## 5. Why ML beats a fixed rule here

A fixed rule such as stale × visible uses a small number of hard thresholds and may miss interactions among several observable signals. ML could help learn a more flexible ranking from multiple pre-outcome signals, although the fixed rule remains an important baseline. The goal is to improve the review priority decision, not to assume that ML will always perform better.

In [14]:
features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

print("Candidate observable signals:", len(features))
print("Signals:", features)
print("Baseline rule: stale × visible × impressions_90d")
print("ML advantage: learn combinations of multiple pre-outcome signals.")

Candidate observable signals: 6
Signals: ['content_age_days', 'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr', 'word_count']
Baseline rule: stale × visible × impressions_90d
ML advantage: learn combinations of multiple pre-outcome signals.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.